*Environment Setup*

In [3]:
# Cell 1: Environment Setup & Imports
!pip install -q yfinance pytz pandas_ta

import pandas as pd
import numpy as np
import yfinance as yf
import pytz
import warnings
import pandas_ta as ta 

# We'll ignore scattered timezone warnings for cleaner output
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"yfinance version: {yf.__version__}")


Libraries imported successfully!
Pandas version: 3.0.1
yfinance version: 1.2.0


In [4]:
# Cell 2: Load sample FNSPID Data (5 Rows)
fnspid_path = 'dl/nasdaq_exteral_data.csv' 

df_sample = pd.read_csv(fnspid_path, nrows=5)

print("Columns in our dataset:")
print(df_sample.columns.tolist())
display(df_sample.head())



Columns in our dataset:
['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url', 'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary']


,Unnamed: 0,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
0,0.0,2023-12-16 23:00:00 UTC,Interesting A Put And Call Options For August ...,A,https://www.nasdaq.com/articles/interesting-a-...,NaN,NaN,"Investors in Agilent Technologies, Inc. (Symbo...",Because the $125.00 strike represents an appro...,The current analytical data (including greeks ...,Below is a chart showing the trailing twelve m...,"At Stock Options Channel, our YieldBoost formu..."
1,1.0,2023-12-12 00:00:00 UTC,Wolfe Research Initiates Coverage of Agilent T...,A,https://www.nasdaq.com/articles/wolfe-research...,NaN,NaN,"Fintel reports that on December 13, 2023, Wolf...","Fintel reports that on December 13, 2023, Wolf...","T. Rowe Price Investment Management holds 10,1...",Agilent Technologies Declares $0.24 Dividend O...,The projected annual revenue for Agilent Techn...
2,2.0,2023-12-12 00:00:00 UTC,Agilent Technologies Reaches Analyst Target Price,A,https://www.nasdaq.com/articles/agilent-techno...,NaN,NaN,"In recent trading, shares of Agilent Technolog...","In recent trading, shares of Agilent Technolog...","In recent trading, shares of Agilent Technolog...",When a stock reaches the target an analyst has...,When a stock reaches the target an analyst has...
3,3.0,2023-12-07 00:00:00 UTC,Agilent (A) Enhances BioTek Cytation C10 With ...,A,https://www.nasdaq.com/articles/agilent-a-enha...,NaN,NaN,Agilent Technologies A is enhancing its BioTek...,"Per a Grand View Research report, the global m...","Notably, Agilent enhanced the BioTek Cytation ...","Agilent Technologies, Inc. Price and Consensus...","Notably, Agilent enhanced the BioTek Cytation ..."
4,4.0,2023-12-07 00:00:00 UTC,"Pre-Market Most Active for Dec 7, 2023 : SQQQ,...",A,https://www.nasdaq.com/articles/pre-market-mos...,NaN,NaN,The NASDAQ 100 Pre-Market Indicator is up 70.2...,ProShares UltraPro Short QQQ (SQQQ) is -0.15 a...,"As reported by Zacks, the current mean recomme...","The total Pre-Market volume is currently 39,23...",The NASDAQ 100 Pre-Market Indicator is up 70.2...


In [5]:
# Cell 3: Loading & Filtering Dataset
from torch._jit_internal import ignore
# We load specific tickers and ignore summary text columns for now

fnspid_path = 'dl/nasdaq_exteral_data.csv' 
# We are focusing on apple, microsoft and tesla
# target_tickers = ['AAPL','MSFT','TSLA']
target_tickers = ['MSFT', 'DIS', 'WMT']

print(f"Loading data for {target_tickers}...")

# Process in 500k blocks
chunks = []
for chunk in pd.read_csv(fnspid_path, usecols=['Date', 'Article_title', 'Stock_symbol'], chunksize=500_000, dtype=str, on_bad_lines='skip'):
    # Filter
    filtered_chunk = chunk[chunk['Stock_symbol'].isin(target_tickers)]
    chunks.append(filtered_chunk)

# Combine chunks into main dataframe
df_news = pd.concat(chunks, ignore_index=True)

# Rename columns
df_news.rename(columns={'Stock_symbol': 'Ticker', 'Article_title': 'Headline'}, inplace=True)

# Convert Date string into real datetime objects
df_news['Date'] = pd.to_datetime(df_news['Date'], errors='coerce')

# Drop rows missing crucial info
df_news.dropna(subset=['Date', 'Headline'], inplace=True)

# Sort chronologically
df_news.sort_values(by=["Ticker", 'Date'], inplace=True)
df_news.reset_index(drop=True, inplace=True)

print(f"Success! Loaded {len(df_news):,} relevant headlines.")
display(df_news.head())


Loading data for ['MSFT', 'DIS', 'WMT']...
Success! Loaded 27,077 relevant headlines.


,Date,Headline,Ticker
0,2019-06-13 00:00:00+00:00,Analyst: Demand for Disney+ Will Be Stronger T...,DIS
1,2019-06-14 00:00:00+00:00,"Roku Stock Is Streaming Profits for Investors,...",DIS
2,2019-06-14 00:00:00+00:00,Buy Netflix Stock Because NFLX Continues to Wi...,DIS
3,2019-06-14 00:00:00+00:00,"Friday’s Vital Data: Lululemon, Disney and Snap",DIS
4,2019-06-14 00:00:00+00:00,"3 Big Stock Charts for Friday: Symantec, Verte...",DIS


In [6]:
# Check headline count
print("Headline count per ticker:")
print(df_news['Ticker'].value_counts())

Headline count per ticker:
Ticker
DIS     9654
MSFT    8737
WMT     8686
Name: count, dtype: int64


In [6]:
# Cell 4: Timezone alignment & Weekend shift
from IPython.core.display_functions import display
print("Aligning timezones and shifting weekends...")

# 1. Convert from UTC to US/Eastern time (NYC market time)
df_news['Date'] = df_news['Date'].dt.tz_convert('US/Eastern')

# 2. Extract calendar day (Exclude specific hour, min) for DAILY open/close price
df_news['Trading_Date'] = df_news['Date'].dt.normalize()

# 3. Handle weekend effect
def shift_to_next_trading_day(d):
    if d.weekday() ==5:
        return d + pd.Timedelta(days=2)
    elif d.weekday() == 6:
        return d + pd.Timedelta(days=1)
    return d

# Apply the shift to our new Trading_Date column
df_news['Trading_Date'] = df_news['Trading_Date'].apply(shift_to_next_trading_day)

# 4. Remove timezone stamp from Trading_Date
df_news['Trading_Date'] = df_news['Trading_Date'].dt.tz_localize(None)

print(f"Time alignment complete! Comparing original UTC date to our mapped Trading Date:")
display(df_news[['Date', 'Trading_Date', 'Ticker', 'Headline']].head(8))

Aligning timezones and shifting weekends...
Time alignment complete! Comparing original UTC date to our mapped Trading Date:


,Date,Trading_Date,Ticker,Headline
0,2019-06-12 20:00:00-04:00,2019-06-12,DIS,Analyst: Demand for Disney+ Will Be Stronger T...
1,2019-06-13 20:00:00-04:00,2019-06-13,DIS,"Roku Stock Is Streaming Profits for Investors,..."
2,2019-06-13 20:00:00-04:00,2019-06-13,DIS,Buy Netflix Stock Because NFLX Continues to Wi...
3,2019-06-13 20:00:00-04:00,2019-06-13,DIS,"Friday’s Vital Data: Lululemon, Disney and Snap"
4,2019-06-13 20:00:00-04:00,2019-06-13,DIS,"3 Big Stock Charts for Friday: Symantec, Verte..."
5,2019-06-13 20:00:00-04:00,2019-06-13,DIS,WarnerMedia Plans to Reach 70 Million Streamin...
6,2019-06-13 20:00:00-04:00,2019-06-13,DIS,Netflix Stock Still Wears the Streaming Crown
7,2019-06-14 20:00:00-04:00,2019-06-14,DIS,Mattel Just Turned Down Another Buyout Offer; ...


In [7]:
# Cell 5: Fix DST & Run FinBERT Sentiment
# Daylight Saving Time anomaly should be fixed (Previous display show the trading date at 1AM) by extracting only date string
df_news['Trading_Date'] = df_news['Trading_Date'].dt.date.astype(str)

print("Loading FinBERT model...")
import torch
from transformers import pipeline
device = 0 if torch.cuda.is_available() else -1

# Load FinBERT
finbert = pipeline(
    'sentiment-analysis',
    model='ProsusAI/finbert',
    device=device,
    truncation=True,
    max_length=512
)

print(f"Scoring {len(df_news)} headlines...")

# Function to map: positive -> +confidence | negative -> -confidence | neutral -> 0
def score_sentiment(label, score):
    if label.lower() == 'positive': return float(score)
    elif label.lower() == "negative": return -float(score)
    else: return 0.0

headlines = df_news['Headline'].tolist()
sentiment_scores = []
batch_size = 64

# Progress bar for visual
from tqdm.auto import tqdm

for i in tqdm(range(0, len(headlines), batch_size), desc = "Scoring Sentiments"):
    batch = headlines[i:i+batch_size]
    results = finbert(batch)
    batch_scores = [score_sentiment(res['label'], res['score']) for res in results]
    sentiment_scores.extend(batch_scores)

# Add continous scores to dataframe
df_news['Sentiment'] = sentiment_scores

# Aggregate: Avg all the news for specific stock on specific day into one feature
daily_sentiment = df_news.groupby(['Ticker', 'Trading_Date'])['Sentiment'].mean().reset_index()

print("Sentiment Aggregation Complete! Daily Sentiment:")
display(daily_sentiment.head(8))


Loading FinBERT model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 16351.27it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Scoring 27077 headlines...


Scoring Sentiments: 100%|██████████| 424/424 [09:18<00:00,  1.32s/it]

Sentiment Aggregation Complete! Daily Sentiment:


,Ticker,Trading_Date,Sentiment
0,DIS,2019-06-12,0.955217
1,DIS,2019-06-13,0.000000
2,DIS,2019-06-14,-0.420206
3,DIS,2019-06-17,0.080462
4,DIS,2019-06-18,0.374311
5,DIS,2019-06-19,-0.323763
6,DIS,2019-06-20,0.000000
7,DIS,2019-06-21,0.000000


In [8]:
# Cell 6: Fetching Market Data with yfinance
print(f"Fetching market data for {target_tickers}...")

# We get start and end dates from news data bounds
start_date = daily_sentiment['Trading_Date'].min()
end_date = daily_sentiment['Trading_Date'].max()

# Convert to standard datetime
start_dt = pd.to_datetime(start_date)

# Add 5-day buffer to end_date, to prepare to predict FUTURE prices (days after news was never published)
end_dt = pd.to_datetime(end_date) + pd.Timedelta(days=5)

# Download data from yfinance
market_data = yf.download(
    target_tickers,
    start=start_dt.strftime('%Y-%m-%d'),
    end=end_dt.strftime('%Y-%m-%d'),
    group_by='ticker',
)

# Unpack yfinance multi-index into flat table
price_records = []
for ticker in target_tickers:
    if len(target_tickers) > 1:
        ticker_df = market_data[ticker].copy()
    else:
        ticker_df = market_data.copy()

    ticker_df['Ticker'] = ticker
    ticker_df.reset_index(inplace=True) 

    if 'Adj Close' in ticker_df.columns:
        ticker_df = ticker_df[['Date', 'Adj Close', 'Volume', 'Ticker']].dropna()
        ticker_df.rename(columns={'Adj Close': 'Close'}, inplace=True)
    else:
        ticker_df = ticker_df[['Date', 'Close', 'Volume', 'Ticker']].dropna()

    price_records.append(ticker_df)

df_prices = pd.concat(price_records, ignore_index=True)

# Format dates to match sentiment data
df_prices['Trading_Date'] = df_prices['Date'].dt.date.astype(str)
df_prices.drop(columns=['Date'], inplace=True)

print(f"Market data fetched successfully! Extracted {len(df_prices)} daily price rows")
display(df_prices.head())

Fetching market data for ['MSFT', 'DIS', 'WMT']...


[*********************100%***********************]  3 of 3 completed

Market data fetched successfully! Extracted 3822 daily price rows


Price,Close,Volume,Ticker,Trading_Date
0,103.517593,33665600,MSFT,2018-11-30
1,104.637802,34732800,MSFT,2018-12-03
2,101.305168,45197000,MSFT,2018-12-04
3,101.930634,49107400,MSFT,2018-12-06
4,97.851173,45044900,MSFT,2018-12-07


In [10]:
import os

# Cell 7: Final Merge & Creating ML Target
print("Merging sentiment and market data...")

# Combine them by matching Ticker and Trading Date
df_final = pd.merge(df_prices, daily_sentiment, on=['Ticker', 'Trading_Date'], how='left')

# Stock trades every weekday, but news doesn't, we fill missing sentiment with neutral (0.0)
df_final['Sentiment'] = df_final['Sentiment'].fillna(0.0)

# Sort strictly by Ticker and Date 
df_final.sort_values(by=['Ticker', 'Trading_Date'], inplace=True)
df_final.reset_index(drop=True, inplace=True)

# calculate SMA and RSI
for ticker in df_final['Ticker'].unique():
    mask = df_final['Ticker' ] == ticker
    
    df_final.loc[mask, 'SMA_20'] = df_final[mask].ta.sma(length=20)
    df_final.loc[mask, 'RSI_14'] = df_final[mask].ta.rsi(length=14)

# first 20 days for every ticker will be NaN becuase need 20 days of history
df_final.dropna(subset=['SMA_20', 'RSI_14'], inplace=True)

# We want to predict tomorrow's movement, so we create 'Next_close' column by shifting the close column up by 1 row within each ticker grouping
df_final['Next_Close'] = df_final.groupby('Ticker')['Close'].shift(-1)

# Calculate price change percentage between today and tomorrow
df_final['Target_Price_Change_Pct'] = (df_final['Next_Close'] - df_final['Close']) / df_final['Close']

# Binary Target: 1 if prices go up | 0 if goes down or stays flat
df_final['Target_Direction'] = (df_final['Target_Price_Change_Pct'] > 0).astype(int)

# Since we shifted data up by 1, the last chronological day wont have a Next_Close, we will drop those NA rows to prevent errors.
df_final.dropna(subset=['Next_Close'], inplace=True)

print("Pipeliine Complete! Final Dataset:")
display(df_final.head(10))

# Save it
base_dir = "datasets"

print(f"Exporting individual datasets to '{base_dir}/'...")

for ticker in df_final['Ticker'].unique():
    ticker_dir = os.path.join(base_dir, ticker)
    os.makedirs(ticker_dir, exist_ok=True)
    
    ticker_df = df_final[df_final['Ticker'] == ticker].copy()
    
    file_path = os.path.join(ticker_dir, f"{ticker}_DatasetV1.csv")
    
    # 5. Save the CSV
    ticker_df.to_csv(file_path, index=False)
    print(f" - Saved {len(ticker_df)} rows to: {file_path}")

Merging sentiment and market data...
Pipeliine Complete! Final Dataset:


,Close,Volume,Ticker,Trading_Date,Sentiment,SMA_20,RSI_14,Next_Close,Target_Price_Change_Pct,Target_Direction
19,105.833809,7241700,DIS,2018-12-31,0.0,106.107507,50.913971,105.177475,-0.006202,0
20,105.177475,9723500,DIS,2019-01-02,0.0,105.835753,49.097412,102.629349,-0.024227,0
21,102.629349,10594700,DIS,2019-01-03,0.0,105.424621,42.724089,105.795197,0.030847,1
22,105.795197,10122800,DIS,2019-01-04,0.0,105.309220,51.199909,106.712135,0.008667,1
23,106.712135,6714700,DIS,2019-01-07,0.0,105.169750,53.353029,107.542206,0.007779,1
24,107.542206,8730700,DIS,2019-01-08,0.0,105.142725,55.276742,108.748703,0.011219,1
25,108.748703,5931900,DIS,2019-01-09,0.0,105.181815,57.988677,108.874161,0.001154,1
26,108.874161,6160100,DIS,2019-01-10,0.0,105.221870,58.272031,108.729401,-0.001330,0
27,108.729401,4819100,DIS,2019-01-11,0.0,105.243105,57.787709,108.507401,-0.002042,0
28,108.507401,6980600,DIS,2019-01-14,0.0,105.196293,57.005221,107.870361,-0.005871,0


Exporting individual datasets to 'datasets/'...
 - Saved 1254 rows to: datasets\DIS\DIS_DatasetV1.csv
 - Saved 1254 rows to: datasets\MSFT\MSFT_DatasetV1.csv
 - Saved 1254 rows to: datasets\WMT\WMT_DatasetV1.csv


In [1]:
import os
import pandas as pd

# Path to your final datasets
dataset_path = "datasets/"
tickers = ['MSFT', 'DIS', 'WMT']

# 1. Get headline counts from our original news dataframe
# This works since df_news is already in your notebook's memory
headline_counts = df_news['Ticker'].value_counts()

print(f"{'Ticker':<10} | {'Trading Days':<15} | {'Total Headlines':<20}")
print("-" * 55)

for ticker in tickers:
    file_path = os.path.join(dataset_path, ticker, f"{ticker}_DatasetV1.csv")
    if os.path.exists(file_path):
        df_temp = pd.read_csv(file_path)
        
        # 2. Extract headline count for this ticker
        h_count = headline_counts.get(ticker, 0)
        
        print(f"{ticker:<10} | {len(df_temp):<15} | {h_count:<20,}")
    else:
        print(f"{ticker:<10} | File Not Found")

print("-" * 55)


NameError: name 'df_news' is not defined